# 01 - Inlet profile validation

1. **Directional design speed** (NBR 6123 / EN 1991-1-4) from the
   wind-analysis CSVs -- no simulation data needed.
2. **ABL profile validation** per terrain category -- mean velocity + TI vs
   code, spectrum, integral length.

ABL probe lines are read directly from the h5 files (no `nassu`). Results are
shown inline AND written to `deliverables/<version>/inflow/`.

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from cfdmod import plot_config
from cfdmod.report import DebugWriter
from cfdmod.analytical import WindProfile_NBR, WindProfile_EU
from cfdmod.inflow import InflowData, NormalizationParameters
from cfdmod import inflow_report as ir

import logging
logging.getLogger("cfdmod").setLevel(logging.WARNING)
plot_config.apply_style()

project = pathlib.Path("/data/eng/consulting/NNN_CaseName")  # <- set the case root
CASE = "CaseName"  # <- results case-name prefix: results/<batch>/<CASE>_<dir>/...
case_data = project / "post_processing/pp_config/case_data"
results   = project / "results"
dct_case = json.loads((case_data / "global_data.json").read_text())
batch = dct_case["analysis"]["batch_name"]
H, V0 = float(dct_case["H"]), float(dct_case["V0"])
cats = sorted(k.split("directions_cat")[1] for k in dct_case["analysis"] if k.startswith("directions_cat"))
VERSION = "v3"
dbg = DebugWriter(pathlib.Path.cwd(), stage="inflow", version=VERSION)
print(f"H={H} V0={V0} | terrain categories {cats}")
print("deliverables ->", dbg.deliverables_dir)

## 1. Directional design speed (NBR + EU)

In [ ]:
wp_nbr = WindProfile_NBR.build(case_data / "wind_analysis_NBR.csv", V0=V0)
wp_eu  = WindProfile_EU.build(case_data / "wind_analysis_EU.csv", Vb=V0)

u_nbr = ir.directional_reference_speed(wp_nbr, height=H, recurrence_period=50, use_kd=True)
u_eu  = ir.directional_reference_speed(wp_eu,  height=H, recurrence_period=50, use_kd=True)
print(f"governing U_H @ {H:g} m: NBR {u_nbr.max():.2f} @ {u_nbr.idxmax():g} deg "
      f"| EU {u_eu.max():.2f} @ {u_eu.idxmax():g} deg")

fig, ax = plot_config.new_axes(xlabel="wind direction [deg]", ylabel="U_H [m/s]",
                               title=f"Directional design speed @ H={H:g} m")
ax.plot(u_nbr.index, u_nbr.to_numpy(), "-o", ms=3, label="NBR 6123")
ax.plot(u_eu.index, u_eu.to_numpy(), "-s", ms=3, label="EN 1991-1-4")
ax.legend()
dbg.savefig(fig, "directional_U_H.png", deliverable=True)
plt.show()
table = u_nbr.rename("U_H_NBR").to_frame().join(u_eu.rename("U_H_EU"))
dbg.save_csv(table.rename_axis("wind_direction").reset_index().round(3), "directional_U_H.csv", deliverable=True)
table.round(2)

## 2. ABL profile validation (nassu-free probe read)

Set `REP` per terrain category: representative direction (matching a
`<CASE>_noBody_<dir>` run on disk) and EU/NBR terrain-category labels.

In [ ]:
def read_line(h5_path):
    parts = []
    with pd.HDFStore(h5_path, mode="r") as store:
        for key in store.keys():
            parts.append(pd.read_hdf(store, key=key))
    return pd.concat(parts).reset_index(drop=True)


def build_inflow(direction, line="line0", component="ux"):
    folder = (results / batch / f"{CASE}_noBody_{direction}" / "000/probes"
              / "hist_series" / "abl_profile")
    wide = read_line(folder / f"line.{line}.{component}.h5")
    num = [c for c in wide.columns if str(c).isnumeric()]
    long = wide.melt(id_vars=["time_step"], value_vars=num,
                     var_name="point_idx", value_name=component)
    long["point_idx"] = long["point_idx"].astype(int)
    points = pd.read_csv(folder / f"line.{line}.points.csv")
    return InflowData(data=long, points=points), folder

In [ ]:
# representative direction + EU/NBR category labels per terrain category
REP = {"0": ("000.0", "0", "I"), "3": ("022.5", "III", "III")}

measured_u_ref = {}
for c in cats:
    rep_dir, cat_eu, cat_nbr = REP.get(c, (dct_case["analysis"][f"directions_cat{c}"][0], "III", "III"))
    try:
        inflow, folder = build_inflow(rep_dir)
        prof = ir.detect_profiles(inflow, min_points=3)[0]
        u_ref = ir.reference_velocity(prof, inflow, H)
        L = ir.integral_length_scale(inflow, prof.nearest_index(H), u_ref)
        print(f"[cat{c}] dir {rep_dir}: u_ref(H)={u_ref:.2f} m/s | L(H)={L:.1f} m")
        measured_u_ref[c] = float(u_ref)
        norm = NormalizationParameters(reference_velocity=max(u_ref, 1e-6), characteristic_length=H)
        fig, _ = ir.plot_profile_vs_code(prof, inflow, H, cat_eu=cat_eu, cat_nbr=cat_nbr)
        dbg.savefig(fig, f"profile_vs_code_cat{c}.png", deliverable=True); plt.show()
        fig = ir.plot_spectrum(prof, inflow, H, norm)
        dbg.savefig(fig, f"spectrum_cat{c}.png"); plt.show()
        L_num = ir.integral_length_scale_profile(inflow, prof)
        fig, _ = ir.plot_integral_length_scale(prof.z, L_num, H)
        dbg.savefig(fig, f"integral_length_cat{c}.png"); plt.show()
    except Exception as e:
        print(f"[cat{c}] ABL validation skipped ({type(e).__name__}: {e})")

## Handoff to the load notebooks (`_shared.json`)

In [ ]:
shared = {
    "H": H, "V0": V0,
    "governing_design_U_H": {"NBR": float(u_nbr.max()), "EU": float(u_eu.max())},
    "design_U_H_by_direction_NBR": {f"{float(d):g}": float(v) for d, v in u_nbr.items()},
    "measured_u_ref_by_category": measured_u_ref,
}
out = pathlib.Path.cwd() / "_shared.json"
out.write_text(json.dumps(shared, indent=2))
print("wrote", out); print("deliverables ->", dbg.deliverables_dir)
shared